# Agent loop from scratch: bill split and tip

We build the same **tool loop** pattern as Week 1 (`5_extra.ipynb`): define Python functions, expose them as tools, run the chat API in a loop until the model answers without calling tools.

**Problem:** From a bill amount, tip percentage, and headcount, figure out **each person’s share** after tip. The model must use **`calculate`** for math (no mental arithmetic) and **todos** to plan and track steps.

This contribution implements the an agent loop from scratch with todos + a safe calculate tool. The scenario (bill + tip + split) is intentionally simple so it clearly demonstrates the agent loop—tool schemas, dispatch, todos, and the driver loop. A natural follow-up would be the same setup but solving a problem worth using LLMs.

## 1. Imports and client

In [ ]:
import ast
import json
import operator
import os

from dotenv import load_dotenv
from openai import OpenAI
from rich.console import Console

load_dotenv(override=True)

In [ ]:
def show(text: str | None) -> None:
    if text is None:
        return
    try:
        Console().print(text)
    except Exception:
        print(text)

In [ ]:
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
openai = OpenAI()

## 2. Todo state (plan + execute)

Parallel lists keep task text and completion flags aligned.

In [ ]:
todos: list[str] = []
completed: list[bool] = []


def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result


def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()


def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

## 3. Calculator tool (deterministic math)

Only safe numeric expressions are evaluated—no `import`, names, or calls.

In [ ]:
_BINOPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.Mod: operator.mod,
}


def _eval_node(node: ast.AST) -> float:
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return float(node.value)
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
        return -_eval_node(node.operand)
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.UAdd):
        return _eval_node(node.operand)
    if isinstance(node, ast.BinOp):
        op = _BINOPS.get(type(node.op))
        if op is None:
            raise ValueError("Operator not allowed")
        return op(_eval_node(node.left), _eval_node(node.right))
    raise ValueError("Expression not allowed")


def calculate(expression: str) -> str:
    """Evaluate a numeric expression like (120 * 1.18) / 5; returns a short string result."""
    tree = ast.parse(expression.strip(), mode="eval")
    value = _eval_node(tree.body)
    rounded = round(value, 2)
    if rounded == int(rounded):
        return str(int(rounded))
    return f"{rounded:.2f}"

## 4. Tool schemas (OpenAI function-calling format)

In [ ]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                "type": "array",
                "items": {"type": "string"},
                "title": "Descriptions",
            }
        },
        "required": ["descriptions"],
        "additionalProperties": False,
    },
}

mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "index": {
                "description": "The 1-based index of the todo to mark as complete",
                "title": "Index",
                "type": "integer",
            },
            "completion_notes": {
                "description": "Notes about how you completed the todo in rich console markup",
                "title": "Completion Notes",
                "type": "string",
            },
        },
        "required": ["index", "completion_notes"],
        "additionalProperties": False,
    },
}

calculate_json = {
    "name": "calculate",
    "description": "Evaluate a numeric arithmetic expression (+ - * / ** % parentheses, decimals). Use for bill and tip math—do not guess.",
    "parameters": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": 'Expression only, e.g. "120 * 1.18" or "(120 * 1.18) / 5"',
            }
        },
        "required": ["expression"],
        "additionalProperties": False,
    },
}

tools = [
    {"type": "function", "function": create_todos_json},
    {"type": "function", "function": mark_complete_json},
    {"type": "function", "function": calculate_json},
]

## 5. Dispatch and driver loop

`max_iterations` avoids a runaway loop if the model keeps calling tools.

In [ ]:
def handle_tool_calls(tool_calls) -> list[dict]:
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if callable(tool) else {}
        results.append(
            {"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id}
        )
    return results


def loop(messages: list[dict], max_iterations: int = 32) -> None:
    iteration = 0
    response = None
    while iteration < max_iterations:
        iteration += 1
        response = openai.chat.completions.create(
            model=MODEL, messages=messages, tools=tools
        )
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            if not tool_calls:
                break
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            break
    else:
        raise RuntimeError(f"Stopped after {max_iterations} iterations (possible loop).")
    if response is None:
        return
    content = response.choices[0].message.content
    show(content)

## 6. Demo: split the bill

Run this after setting `OPENAI_API_KEY`. The agent should create todos, use `calculate` for totals and per-person amount, mark todos done, then answer in Rich markup.

In [ ]:
system_message = """
You solve the user's question by planning with todo tools, then executing each step.
Use calculate for every arithmetic step (totals, tip, splits). Do not do math in prose without calling calculate.
When done, reply with the final answer in Rich console markup (no code fences).
Do not ask the user for clarification; use only the numbers given.
"""

user_message = """
The restaurant bill is $120. We want to add an 18% tip and split the total evenly among 5 people.
How much does each person pay? Give the amount in dollars.
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message},
]

todos, completed = [], []
loop(messages)